In [1]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dask import delayed, compute
from tqdm import tqdm
from dask.distributed import Client, LocalCluster

# ============================
# User settings
# ============================

# Generator site list
# gen_csv = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv")
# cluster_dates = pd.read_csv("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/alpine_cluster.csv")

boco_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_boco_wind_vec.nc"
clust_ds = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/C2_cluster_wind_vec.nc"

# Dask cluster

client = Client()
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.08/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35419 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/35419/status,
Dashboard: /proxy/35419/status,Workers: 7
Total threads: 7,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42631,Workers: 0
Dashboard: /proxy/35419/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:46359,Total threads: 1
Dashboard: /proxy/33031/status,Memory: 4.57 GiB
Nanny: tcp://127.0.0.1:40155,


In [2]:
ds = xr.open_dataset(boco_ds, chunks={'time':1}, engine='h5netcdf')
ds

OSError: Unable to synchronously open file (file signature not found)

In [9]:
@delayed
def plot_frame(u, v, lat, lon, t, output_dir, quiver_scale, extent):
    
    plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.STATES, linestyle=':')
    
    plt.contourf(lon, lat, speed, cmap='viridis', transform=ccrs.PlateCarree())
    plt.colorbar(label='Wind speed (m/s)')
    plt.quiver(lon, lat, u, v, scale=quiver_scale, color='white', transform=ccrs.PlateCarree())
    plt.title(f'Wind vectors at {str(t)}')
    
    dt_str = np.datetime_as_string(t, unit='m')
    dt_str = dt_str.replace('-', '')[2:8] + '_' + dt_str[11:13] + dt_str[14:16]
    filename = os.path.join(output_dir, f'wind_{dt_str}.png')
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    return filename

# -------------------------------
# Function to save all frames for a given hour or all times
# -------------------------------
def save_wind_frames(ds, u_var='ua100m', v_var='va100m', hour=None, 
                     output_dir='wind_plots', step=5, quiver_scale=150, parallel=True, extent=None):
    """
    Save plots of all entries at a specific hour (or all time steps if hour=None).
    """
    # Select times
    if hour is not None:
        ds_sub = ds.sel(time=ds.time.dt.hour == hour)
    else:
        ds_sub = ds
    
    tasks = []
    for i in range(len(ds_sub.time)):
        u = ds_sub[u_var].isel(time=i)[::step, ::step]
        v = ds_sub[v_var].isel(time=i)[::step, ::step]
        lat = ds_sub['lat'][::step].values
        lon = ds_sub['lon'][::step].values
        t = ds_sub.time[i].values

        if parallel:
            tasks.append(plot_frame(u, v, lat, lon, t, output_dir, quiver_scale, extent))
        else:
            # For serial execution, just call the delayed function and compute immediately
            plot_frame(u, v, lat, lon, t, output_dir, quiver_scale, extent).compute()
    
    if parallel:
        results = compute(*tasks)
        return results
    return None

In [ ]:
# Local Flows
save_wind_frames(ds,
                 hour=17,
                 output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/C2_wind_hour17_zoom',
                 step=3,
                 quiver_scale=400,
                 parallel=True,
                 extent=[145, 151, -38, -33])

In [5]:
# save_wind_frames(ds,
#                  hour=12,
#                  output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/wind_hour_12',
#                  step=2,
#                  quiver_scale=400,
#                  parallel=True,
#                  extent=[147, 152, -40, -34])

In [6]:
# save_wind_frames(ds,
#                  hour=6,
#                  output_dir='/g/data/ng72/ms5578/ID_HW_BARRA/data/output/alpine_wind/wind_hour_6',
#                  step=2,
#                  quiver_scale=400,
#                  parallel=True,
#                  extent=[145, 153, -40, -33])